# ReLoG - Tower Depth ablation

In [1]:
# IMPORT

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import copy
import random
import math

In [ ]:
# =============================================================================
# SEED
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

df_sampled      = pd.read_parquet('../../preprocessing/CD_Vinyl_review.parquet')
df_meta_aligned = pd.read_parquet('../../preprocessing/CD_Vinyl_meta.parquet')

NUM_USERS    = df_sampled['user_id_int'].nunique()
NUM_ITEMS    = df_sampled['item_id_int'].nunique()
INTERACTIONS = len(df_sampled)
SPARSITY     = (1 - (INTERACTIONS / (NUM_USERS * NUM_ITEMS))) * 100

print(f"Utenti: {NUM_USERS}, Item: {NUM_ITEMS}, Interactions: {INTERACTIONS}")
print(f"Sparsity: {SPARSITY:.2f}%")

Utenti: 3905, Item: 27149, Interactions: 49659
Sparsity: 99.95%


In [ ]:
# =============================================================================
# EMBEDDINGS SBERT
# =============================================================================

sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

df_sampled['summary']    = df_sampled['summary'].fillna('')
df_sampled['reviewText'] = df_sampled['reviewText'].fillna('')
df_sampled['full_text']  = df_sampled['summary'] + ". " + df_sampled['reviewText']

print("Compute review embeddings...")
review_embeddings = sbert.encode(
    df_sampled['full_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
review_emb_map = {i: review_embeddings[i] for i in range(len(df_sampled))}

print("Compute embeddings item metadata...")
meta_embeddings = sbert.encode(
    df_meta_aligned['meta_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
item_meta_tensor = torch.tensor(meta_embeddings, dtype=torch.float32)
print(f"Item bank: {item_meta_tensor.shape}")

item_meta_tensor_gpu = item_meta_tensor.to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolo embeddings review...


Batches:   0%|          | 0/776 [00:00<?, ?it/s]

Calcolo embeddings metadati item...


Batches:   0%|          | 0/425 [00:00<?, ?it/s]

Item bank: torch.Size([27149, 384])


##  Architecture

In [ ]:
class ParametricTower(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256, num_layers=2):
        super().__init__()
        layers = []
        last_dim = input_dim
        
        for i in range(num_layers - 1):
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim
        
        layers.append(nn.Linear(last_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: [batch, input_dim]
        return F.normalize(self.net(x), dim=-1)

class UserTower(ParametricTower):
    pass

class ItemTower(ParametricTower):
    pass

class LocalScoreFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_emb, item_emb):
        if user_emb.shape[0] != item_emb.shape[0]:
            if user_emb.shape[0] == 1:
                user_emb = user_emb.expand(item_emb.shape[0], -1)
            else:
                raise RuntimeError(f"Shape mismatch: {user_emb.shape} vs {item_emb.shape}")
        x = torch.cat([user_emb, item_emb], dim=-1)
        return self.net(x)


class TwoTowerRecommender(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256, num_layers=2, inference_temperature=0.07):
        super().__init__()
        self.item_tower = ItemTower(input_dim, hidden_dim, output_dim, num_layers)
        self.user_tower = UserTower(input_dim, hidden_dim, output_dim, num_layers)
        self.client_mlp = LocalScoreFunction(input_dim=output_dim * 2, hidden_dim=128)
        self.inference_temperature = inference_temperature

    def get_user_repr(self, review_embeddings):
        return self.user_tower(review_embeddings).mean(dim=0, keepdim=True)

    def get_item_repr(self, item_meta_embeddings):
        return self.item_tower(item_meta_embeddings)

    def training_score(self, user_repr, item_reprs):
        raw_scores = self.client_mlp(user_repr, item_reprs).squeeze(-1)
        return raw_scores / self.inference_temperature

    def score(self, user_repr, item_reprs):
        return self.training_score(user_repr, item_reprs)


def bpr_loss(pos_scores, neg_scores):
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()

In [7]:
def get_client_data(user_id, df, emb_map, mode="train"):
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if len(user_df) < 3:
        return None, None
    indices = user_df.index.tolist()
    train_idx = indices[:-2]
    val_idx   = indices[-2]
    test_idx  = indices[-1]
    X_train = torch.tensor(
        np.array([emb_map[i] for i in train_idx]),
        dtype=torch.float32
    )
    train_item_ids = user_df.loc[train_idx, 'item_id_int'].tolist()
    if mode == "val":
        target_id = int(user_df.loc[val_idx, 'item_id_int'])
    elif mode == "test":
        target_id = int(user_df.loc[test_idx, 'item_id_int'])
    else:
        target_id = None
    return (X_train, train_item_ids), target_id

In [ ]:
def sample_hard_negatives(user_repr, pos_set, all_metas, local_model,
                          num_neg, num_candidates, device, num_total_items):
    all_ids = np.arange(num_total_items)
    pos_arr = np.array(list(pos_set), dtype=np.int64)
    mask = np.ones(num_total_items, dtype=bool)
    mask[pos_arr] = False
    eligible = all_ids[mask]

    if len(eligible) < num_neg:
        return eligible.tolist()

    n_cands = min(num_candidates, len(eligible))
    candidate_ids = np.random.choice(eligible, size=n_cands, replace=False)

    with torch.no_grad():
        cand_ids_tensor = torch.tensor(candidate_ids, device=device)
        cand_metas = all_metas[cand_ids_tensor]
        cand_reprs = local_model.get_item_repr(cand_metas)
        cand_scores = local_model.training_score(user_repr.detach(), cand_reprs)

    top_k = min(num_neg, len(candidate_ids))
    top_indices = torch.topk(cand_scores, top_k).indices.cpu().numpy()
    return candidate_ids[top_indices].tolist()

## Train client

In [ ]:
def get_lr(base_lr, current_step, warmup_steps, total_steps):
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

def train_client(user_id, global_state_dict, X_train_reviews, train_item_ids,
                 all_metas_gpu,   # [OPT-1] riceve direttamente il tensore già su GPU
                 device, client_states,
                 lr=0.001, epochs=5, num_neg=10, use_hard_negatives=True,
                 lr_warmup_steps=10, current_step=0, total_steps=100, num_layers=2):

    local_model = TwoTowerRecommender(num_layers=num_layers).to(device)
    local_model.load_state_dict(global_state_dict, strict=False)

    user_local_data = client_states.get(user_id, None)
    if user_local_data is not None:
        local_model.client_mlp.load_state_dict(user_local_data)

    effective_lr = get_lr(lr, current_step, lr_warmup_steps, total_steps)
    optimizer = torch.optim.Adam(local_model.parameters(), lr=effective_lr)
    local_model.train()

    X_train = X_train_reviews.to(device)
    pos_set = set(train_item_ids)
    num_total_items = all_metas_gpu.shape[0]

    
    pos_tensor = torch.tensor(train_item_ids, device=device)
    pos_metas  = all_metas_gpu[pos_tensor]   

    loss = None
    for _ in range(epochs):
        optimizer.zero_grad()
        user_repr = local_model.get_user_repr(X_train)

        pos_reprs = local_model.get_item_repr(pos_metas)

        if use_hard_negatives:
            neg_ids = sample_hard_negatives(
                user_repr, pos_set, all_metas_gpu, local_model,
                num_neg=len(train_item_ids) * num_neg,
                num_candidates=2000, device=device,
                num_total_items=num_total_items
            )
        else:
            all_ids = np.arange(num_total_items)
            mask = np.ones(num_total_items, dtype=bool)
            mask[np.array(list(pos_set))] = False
            neg_ids = np.random.choice(
                all_ids[mask],
                size=len(train_item_ids) * num_neg,
                replace=True
            ).tolist()

        neg_tensor = torch.tensor(neg_ids, device=device)
        neg_metas  = all_metas_gpu[neg_tensor]
        neg_reprs  = local_model.get_item_repr(neg_metas)

        pos_scores = local_model.training_score(user_repr, pos_reprs)
        neg_scores = local_model.training_score(user_repr, neg_reprs)

        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
        optimizer.step()

    client_states[user_id] = local_model.client_mlp.state_dict()

    shared_state = {k: v.cpu() for k, v in local_model.state_dict().items()
                    if 'client_mlp' not in k}

    return shared_state, loss.item(), len(train_item_ids)

In [ ]:
def weighted_fedavg_momentum(global_model, local_weights_list, local_sizes,
                             momentum_buffer, beta=0.9):
    total_samples = sum(local_sizes)
    global_dict   = global_model.state_dict()
    keys_to_agg   = [k for k in global_dict if 'client_mlp' not in k]

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(global_dict[k]) for k in keys_to_agg}

    with torch.no_grad():
        for key in keys_to_agg:
            layer_avg = torch.zeros_like(global_dict[key])
            for i, w in enumerate(local_weights_list):
                layer_avg.add_(w[key].to(layer_avg.device),
                               alpha=local_sizes[i] / total_samples)

            delta = layer_avg - global_dict[key]
            momentum_buffer[key].mul_(beta).add_(delta, alpha=1 - beta)
            global_dict[key].add_(momentum_buffer[key])

    global_model.load_state_dict(global_dict, strict=False)
    return momentum_buffer

## Validation

In [ ]:
def evaluate_top_k(global_model, eval_users, df, emb_map,
                   all_metas_gpu,   # [OPT-1] tensore già su GPU
                   client_states, k=10, device='cuda', mode="test",
                   eval_fraction=1.0, num_layers=2,):

    if eval_fraction < 1.0:
        n_sample   = max(1, int(len(eval_users) * eval_fraction))
        eval_users = np.random.choice(eval_users, n_sample, replace=False)
    
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)

    global_model.eval()
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)  # [N_items, output_dim]

    hits, ndcgs, count = 0, 0, 0

    use_amp = (device == 'cuda')

    for user_id in tqdm(eval_users, desc=f"Evaluating ({mode})"):
        train_data, target_id = get_client_data(user_id, df, emb_map, mode=mode)
        if train_data is None:
            continue
        X_train, train_ids = train_data

        local_model = TwoTowerRecommender(num_layers=num_layers).to(device)
        local_model.load_state_dict(global_state, strict=False)

        user_local_data = client_states.get(user_id, None)
        if user_local_data is not None:
            local_model.client_mlp.load_state_dict(user_local_data)

        lr_eval = 0.005

        # --- LOCAL FINETUNING ---
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr_eval)
        X_train_dev = X_train.to(device)

        batch_pos = train_ids if len(train_ids) < 32 else random.sample(train_ids, 32)
        pos_t = torch.tensor(batch_pos, device=device)

        train_ids_set = set(train_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(train_ids_set))]

        
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            for _ in range(3):
                optimizer.zero_grad()
                user_repr = local_model.get_user_repr(X_train_dev)
                pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])

                neg_idx = np.random.choice(eligible_neg, size=len(batch_pos), replace=False)
                neg_t   = torch.tensor(neg_idx, device=device)
                neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])

                loss = bpr_loss(
                    local_model.training_score(user_repr, pos_reprs),
                    local_model.training_score(user_repr, neg_reprs)
                )
                loss.backward()
                optimizer.step()

        local_model.eval()
        with torch.no_grad():
            user_repr = local_model.get_user_repr(X_train_dev)

            neg_cands   = list(all_ids_set - train_ids_set - {target_id})
            neg_arr     = np.array(neg_cands)
            neg_embs    = all_item_embs[neg_arr]  # shape: [N_neg, output_dim]
            target_emb  = all_item_embs[target_id].unsqueeze(0)  # [1, output_dim]

            
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)

            all_scores = torch.cat([
                torch.tensor([target_score], device=device),
                neg_scores
            ])
            top_k_idx = torch.topk(all_scores, k).indices.cpu().numpy()

            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1

    if count == 0:
        return 0.0, 0.0
    return hits / count, ndcgs / count

In [ ]:
def get_fewshot_data(user_id, df, emb_map, num_shots):
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
 
    if num_shots is None:
        if len(user_df) < 2:
            return None, None, None
        indices       = user_df.index.tolist()
        shot_idx      = indices[:-2] 
        target_idx    = indices[-1]
    else:
        min_required = num_shots + 1
        if len(user_df) < min_required:
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:num_shots]
        target_idx = indices[num_shots]  
 
    X_shots = torch.tensor(
        np.array([emb_map[i] for i in shot_idx]),
        dtype=torch.float32
    )
    shot_item_ids = user_df.loc[shot_idx, 'item_id_int'].tolist()
    target_id     = int(user_df.loc[target_idx, 'item_id_int'])
 
    return X_shots, shot_item_ids, target_id
 
 
def evaluate_fewshot(global_model, unseen_users, df, emb_map,
                     all_metas_gpu, num_shots,
                     k=10, device='cuda', finetune_epochs=5, lr=0.01, num_layers=2,):
    label = f"{num_shots}-shot" if num_shots is not None else "full"
 
    num_items    = all_metas_gpu.shape[0]
    global_state = global_model.state_dict()
    all_ids_set  = set(range(num_items))
    all_ids_arr  = np.arange(num_items)
    use_amp      = (device == 'cuda')
 
    global_model.eval()
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_metas_gpu[i:i + chunk])
            for i in range(0, num_items, chunk)
        ], dim=0)
 
    hits, ndcgs, count = 0, 0, 0
 
    for user_id in tqdm(unseen_users, desc=f"Few-shot eval ({label})", leave=False):
        X_shots, shot_item_ids, target_id = get_fewshot_data(
            user_id, df, emb_map, num_shots
        )
        if X_shots is None:
            continue
 
        local_model = TwoTowerRecommender(num_layers=num_layers).to(device)
        local_model.load_state_dict(global_state, strict=False)
 
        X_shots_dev   = X_shots.to(device)
        shot_ids_set  = set(shot_item_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(shot_ids_set))]
 
        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr)
 
        if len(shot_item_ids) > 0 and len(eligible_neg) > 0:
            batch_pos = shot_item_ids if len(shot_item_ids) < 32 \
                        else random.sample(shot_item_ids, 32)
            pos_t = torch.tensor(batch_pos, device=device)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                for _ in range(finetune_epochs):
                    optimizer.zero_grad()
                    user_repr = local_model.get_user_repr(X_shots_dev)
                    pos_reprs = local_model.get_item_repr(all_metas_gpu[pos_t])
                    n_neg     = min(len(batch_pos), len(eligible_neg))
                    neg_idx   = np.random.choice(eligible_neg, size=n_neg, replace=False)
                    neg_t     = torch.tensor(neg_idx, device=device)
                    neg_reprs = local_model.get_item_repr(all_metas_gpu[neg_t])
                    loss = bpr_loss(
                        local_model.training_score(user_repr, pos_reprs),
                        local_model.training_score(user_repr, neg_reprs)
                    )
                    loss.backward()
                    optimizer.step()
 
        local_model.eval()
        with torch.no_grad():
            user_repr  = local_model.get_user_repr(X_shots_dev)
            neg_cands  = list(all_ids_set - shot_ids_set - {target_id})
            neg_arr    = np.array(neg_cands)
            neg_embs   = all_item_embs[neg_arr]
            target_emb = all_item_embs[target_id].unsqueeze(0)
 
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)
 
            all_scores = torch.cat([torch.tensor([target_score], device=device), neg_scores])
            top_k_idx  = torch.topk(all_scores, k).indices.cpu().numpy()
 
            if 0 in top_k_idx:
                hits += 1
                rank = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1
 
    if count == 0:
        return 0.0, 0.0
    print(f"  [{label}] utenti valutati: {count}/{len(unseen_users)}")
    return hits / count, ndcgs / count

## Split users

In [ ]:
def split_users(df, unseen_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    all_users = df['user_id_int'].unique()
    rng.shuffle(all_users)
    n_total = len(all_users)
    n_unseen  = int(n_total * unseen_ratio)
    unseen_users  = all_users[:n_unseen]
    train_users = all_users[n_unseen:]
    return train_users, unseen_users

## Run experiment

In [ ]:
def run_experiment(seed, layers=2):
    print(f"\n===== RUN with seed {seed} and num layer: {layers} =====")
    set_seed(seed)

    train_users, unseen_users = split_users(
        df_sampled, unseen_ratio=0.2, seed=seed
    )
    print(f"Train users: {len(train_users)}")
    print(f"Unseen users:   {len(unseen_users)}")

    LR                    = 0.0005
    LOCAL_EPOCHS          = 3
    NUM_NEG_TRAIN         = 10
    USE_HARD_NEG          = True
    CLIENTS_PER_ROUND     = round(0.05 * len(train_users)) # 0.05
    GLOBAL_ROUNDS         = 100
    EVAL_EVERY            = 5
    INFERENCE_TEMPERATURE = 0.07
    FEDAVG_MOMENTUM       = 0.9
    K                     = 20
    EVAL_FRACTION         = 1
    LR_WARMUP_STEPS       = 10


    client_states   = {user_id: None for user_id in train_users}
    best_val_hr    = 0.0
    best_val_ndcg  = 0.0
    best_state      = None
    best_client_states = None
    momentum_buffer = None
        

    global_model = TwoTowerRecommender(
        num_layers=layers,
        inference_temperature=INFERENCE_TEMPERATURE
    ).to(device)

    print(f"\n=== Inizio Training Federato con seed = {seed} ===")
    print(f"{'Round':<6} | {'Loss':<8} | {'HR@' + str(K):<8} | {'NDCG@' + str(K):<8}")
    print("-" * 45)

    for round_num in range(1, GLOBAL_ROUNDS + 1):
        local_weights = []
        local_sizes   = []
        local_losses  = []

        round_state_dict = global_model.state_dict()

        selected = np.random.choice(train_users, CLIENTS_PER_ROUND, replace=False)
        for user_id in selected:
            train_data, _ = get_client_data(user_id, df_sampled, review_emb_map)
            if train_data is None:
                continue
            X_train, train_item_ids = train_data

            w, loss, n = train_client(
                user_id,
                round_state_dict,       
                X_train,
                train_item_ids,
                item_meta_tensor_gpu,   
                device,
                client_states,
                lr=LR,
                epochs=LOCAL_EPOCHS,
                num_neg=NUM_NEG_TRAIN,
                use_hard_negatives=USE_HARD_NEG,
                lr_warmup_steps=LR_WARMUP_STEPS,
                current_step=round_num - 1,
                total_steps=GLOBAL_ROUNDS,
                num_layers=layers
            )

            local_weights.append(w)
            local_sizes.append(n)
            local_losses.append(loss)

        if not local_weights:
            continue

        momentum_buffer = weighted_fedavg_momentum(
            global_model, local_weights, local_sizes,
            momentum_buffer, beta=FEDAVG_MOMENTUM
        )

        avg_loss = sum(local_losses) / len(local_losses)

        if round_num % EVAL_EVERY == 0:
            val_hr, val_ndcg = evaluate_top_k(
                global_model, train_users, df_sampled,
                review_emb_map, item_meta_tensor_gpu,   
                client_states, k=K, device=device, mode="val",
                eval_fraction=EVAL_FRACTION,
                num_layers=layers
            )

            marker = ""
            if val_hr > best_val_hr:
                best_val_hr        = val_hr
                best_state         = copy.deepcopy(global_model.state_dict())
                best_client_states = copy.deepcopy(client_states)
                marker = "  <- Best"
 
            print(f"{round_num:<6} | {avg_loss:<8.4f} | {val_hr:<10.4f} | {val_ndcg:<10.4f} {marker}")
        else:
            print(f"{round_num:<6} | {avg_loss:<8.4f} |")

    print("\n=== End Training ===")

    global_model.load_state_dict(best_state)
     
    print("\n--- TEST WARM USERS ---")
    warm_hr, warm_ndcg = evaluate_top_k(
        global_model, train_users, df_sampled,
        review_emb_map, item_meta_tensor_gpu,
        best_client_states, k=K, device=device, mode="test",
        num_layers=layers
    )
    print(f"Warm  HR@{K}: {warm_hr:.4f}  |  NDCG@{K}: {warm_ndcg:.4f}")

    print("\n--- TEST UNSEEN USERS (few-shot adaptation) ---")
    shot_configs = [1, 2, 3, None]   # None = full
    fewshot_results = {}
 
    for num_shots in shot_configs:
        label = f"{num_shots}-shot" if num_shots is not None else "full"
        hr, ndcg = evaluate_fewshot(
            global_model, unseen_users, df_sampled,
            review_emb_map, item_meta_tensor_gpu,
            num_shots=num_shots, k=K, device=device,
            finetune_epochs=5, lr=0.005,
            num_layers=layers
        )
        fewshot_results[label] = (hr, ndcg)
        print(f"  {label:<8}  HR@{K}: {hr:.4f}  |  NDCG@{K}: {ndcg:.4f}")
 
    return warm_hr, warm_ndcg, fewshot_results

In [ ]:
seeds      = [0] # 0, 1, 2, 3, 4
NUM_LAYERS_ABLATION   = [4, 6, 8]
shot_labels = ["1-shot", "2-shot", "3-shot", "full"]
 
warm_hrs, warm_ndcgs = [], []
fewshot_hrs  = {l: [] for l in shot_labels}
fewshot_ndcgs = {l: [] for l in shot_labels}

for layers in NUM_LAYERS_ABLATION: 
    for s in seeds:
        warm_hr, warm_ndcg, fewshot_results = run_experiment(s, layers=layers)
        warm_hrs.append(warm_hr)
        warm_ndcgs.append(warm_ndcg)
        for label in shot_labels:
            fewshot_hrs[label].append(fewshot_results[label][0])
            fewshot_ndcgs[label].append(fewshot_results[label][1])
     
    K = 20
     
    print("\n" + "=" * 50)
    print("RESULTS (mean ± std on 5 seed)")
    print("=" * 50)
     
    print(f"\n{'Scenario':<12} | {'HR@'+str(K):<18} | {'NDCG@'+str(K):<18}")
    print("-" * 55)
     
    # Warm users
    m_hr   = np.mean(warm_hrs);   s_hr   = np.std(warm_hrs)
    m_ndcg = np.mean(warm_ndcgs); s_ndcg = np.std(warm_ndcgs)
    print(f"{'warm':<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")
     
    # Few-shot unseen users
    for label in shot_labels:
        m_hr   = np.mean(fewshot_hrs[label]);   s_hr   = np.std(fewshot_hrs[label])
        m_ndcg = np.mean(fewshot_ndcgs[label]); s_ndcg = np.std(fewshot_ndcgs[label])
        print(f"{label:<12} | {m_hr:.4f} ± {s_hr:.4f}   | {m_ndcg:.4f} ± {s_ndcg:.4f}")


===== RUN con seed 0 e num layer: 4 =====
Train users: 3124
Unseen users:   781

=== Inizio Training Federato con seed = 0 ===
Round  | Loss     | HR@20    | NDCG@20 
---------------------------------------------
1      | 0.6950   |
2      | 0.6907   |
3      | 0.6881   |
4      | 0.6856   |


Evaluating (val): 100%|██████████| 3124/3124 [02:05<00:00, 24.95it/s]


5      | 0.6827   | 0.0448     | 0.0169       <- Best
6      | 0.6796   |
7      | 0.6785   |
8      | 0.6770   |
9      | 0.6748   |


Evaluating (val): 100%|██████████| 3124/3124 [02:10<00:00, 23.98it/s]


10     | 0.6734   | 0.0307     | 0.0118     
11     | 0.6682   |
12     | 0.6712   |
13     | 0.6681   |
14     | 0.6688   |


...

Evaluating (val): 100%|██████████| 3124/3124 [03:26<00:00, 15.16it/s]


100    | 0.6934   | 0.0077     | 0.0027     

=== Fine Training ===

--- TEST WARM USERS (ultima interazione) ---


Evaluating (test): 100%|██████████| 3124/3124 [03:24<00:00, 15.26it/s]


Warm  HR@20: 0.1748  |  NDCG@20: 0.0582

--- TEST UNSEEN USERS (few-shot adaptation) ---


KeyboardInterrupt: 